# Day 1 — Extra exercises: measurements, summaries and probability

**For two kinds of reader.** If you **missed Day 1** (after informing the lecturer in advance), this notebook is part of your make-up work. If you attended and want **more practice**, work through any part at your own pace; nothing here is required for you.

**What a make-up submission for Day 1 contains** (see `../day-1-makeup.md`):

- **A1–A3 and B1–B3** of this notebook;
- your own solutions to the core in-class exercises in `../day-1-exercises-morning.ipynb` and `../day-1-exercises-afternoon.ipynb`.

The reflection prompts are optional. **C-problems are optional challenges** for everyone. The make-up submission is graded on the 0–4 in-class scale and counts as your in-class score for the missed day.

**Topics.** Part A: the structure, summaries and plots of a repeated-measurement dataset. Part B: Bayes' rule with two hypotheses, densities versus probabilities, expectation and a first prediction.

**Background.** Read the Day 1 slides first: `../day-1-slides-morning.pdf` (R and JupyterLab) and `../day-1-slides-afternoon.pdf` (probability language and the marble game). Each task below names the slide it builds on.

**How to work.** Open in JupyterLab, select **R (bayes)**, run the cells in order from a fresh kernel. Hints are hidden under **Hint**; try first. Write explanations in your own words in the *Your answer* cells: numbers alone are not enough. All data are **synthetic teaching data**; never report them as real measurements. No Stan today.

**Submitting (make-up).** Submit this notebook with outputs and your written answers, together with the in-class exercises, through the announced route by the announced deadline (normally within one week of the missed day). Discussing ideas is fine; the code and explanations must be your own. You may use AI assistants without declaring it, but you should be able to explain every line.

**R used here.** `tibble()` makes a small table; `|>` reads as "and then"; `mutate()` adds a column; `summarise()` collapses rows to one summary row; `group_by()` makes `summarise()` work per group; `bind_rows()` stacks tables; `count(x)` counts how often each value of `x` occurs. Distributions use the four letters from the slides: `d` (probability or density), `p` (P(X ≤ x)), `q` (quantile), `r` (random draws).

## Setup

Run this cell first.

In [ ]:
options(tidyverse.quiet = TRUE)
library(tidyverse)   # tibble, dplyr and ggplot2
set.seed(2026)
options(width = 100, repr.plot.width = 8, repr.plot.height = 5, repr.plot.bg = "white")

## Part A — A repeated-measurement dataset

A laboratory checks its analytical balance every day by weighing a certified **50.000 g reference mass**. Five operators (O1–O5) each weighed the mass **eight times**: four times in the morning session and four in the afternoon. The file `data/weighing-synthetic.csv` has the columns

- `operator`: anonymous operator ID;
- `replicate`: weighing number 1–8 for that operator;
- `session`: morning (replicates 1–4) or afternoon (5–8);
- `mass_g`: the displayed mass in grams, or `NA` if no reading was obtained.

**The scientific question:** is the balance-plus-operator measurement process accurate (centred on 50.000 g), and how variable is it?

### A1. What is in the data? (25 min)

*Background: the slide "Ten lines of R you will use all day" and the R refresher in the morning exercises; the slide "iid: independent and identically distributed" (afternoon).*

(a) Read the file and report the number of rows, the number of operators and the number of missing readings.
(b) What is the **observational unit**, that is, what does one row describe?
(c) Why are the 40 rows *not* 40 independent pieces of information about the balance? Think about what readings by the same operator have in common.
(d) Create the column `dev_mg`: the deviation from 50 g in **milligrams** (1 g = 1000 mg). Why is it easier to read than `mass_g`?

In [ ]:
w <- read.csv("data/weighing-synthetic.csv")
str(w)

In [ ]:
# (a) nrow(w), n_distinct(w$operator), sum(is.na(w$mass_g))
# (d) w <- w |> mutate(dev_mg = ...)

<details><summary><b>Hint</b> (click to open)</summary>

`nrow(w)` counts rows, `n_distinct(w$operator)` counts distinct operators and `sum(is.na(w$mass_g))` counts missing values. `mutate()` adds a new column.

</details>

**Your answer:** *(write here)*

### A2. Summaries and an unusual value (30 min)

(a) Compute the pooled mean, median and standard deviation (SD) of `dev_mg` over all readings. Use `na.rm = TRUE` so that the missing reading is ignored.
(b) Compute the same summaries for each operator separately, with the number of readings n.
(c) One reading looks very different from all others. Find it, and suggest a plausible explanation by looking at its digits.
(d) *Sensitivity analysis:* recompute the pooled mean, median and SD **without** that reading, keeping the original data `w` unchanged (make a copy `ws` without it). Which summaries change a lot, and which hardly change?

In [ ]:
# (a) w |> summarise(n = sum(!is.na(dev_mg)), mean = mean(dev_mg, na.rm = TRUE), median = ..., sd = ...)
# (b) the same summarise() after group_by(operator)
# (c) w |> arrange(desc(abs(dev_mg))) |> head(3)
# (d) ws <- w |> filter(!(operator == "..." & replicate == ...))   # fill in the reading you found

<details><summary><b>Hint</b> (click to open)</summary>

`arrange(desc(abs(dev_mg)))` sorts the rows from the largest to the smallest distance from 0. Compare the recorded value with the operator's other readings: which digit might have been mistyped?

</details>

**Your answer:** *(write here)*

### A3. Two useful plots (20 min)

Make

(a) a histogram of `dev_mg` for all readings together, with labelled axes and units; and
(b) a plot showing each operator's readings separately, with a reference line at 0 mg. For example jittered points: `geom_jitter()` with `geom_vline(xintercept = 0)`. Use `ws` (without the unusual reading) if the scale becomes unreadable, and say so in the title.

What does the per-operator plot show that the pooled histogram hides? Is there a visible difference between morning and afternoon?

In [ ]:
# (a)
ggplot(w, aes(x = dev_mg)) +
  geom_histogram(bins = 20) +
  labs(title = "...", x = "...", y = "...")

# (b) for example:
# ggplot(ws, aes(x = dev_mg, y = operator, shape = session)) +
#   geom_vline(xintercept = 0, linetype = 2) +
#   geom_jitter(height = 0.15, width = 0) +
#   labs(...)

<details><summary><b>Hint</b> (click to open)</summary>

`shape = session` inside `aes()` gives morning and afternoon readings different symbols. ggplot2 warns that it removed the row with the missing reading: mention it in your answer.

</details>

**Your answer:** *(write here)*

### Reflection A (optional)

In 80–120 words, distinguish three things: **variability between weighings by the same operator**, **differences between operators**, and **uncertainty about the balance's true offset**. Which of these would more weighings by the same five operators reduce?

**Your answer:** *(write here)*

## Part B — Probability and updating

### B1. Which machine? Bayes' rule with two hypotheses (30 min)

*Background: the slides "Turning p(data | bag) into p(bag | data)", "We draw blue, white, blue. Which bag?" and "The fourth draw is blue: today's posterior is the new prior" (marble game, afternoon). Here there are two "bags" (machines) instead of five.*

A workshop receives parts from two machines:

| | share of all parts (prior) | P(defective \| machine) |
|---|---|---|
| machine A | 0.70 | 0.02 |
| machine B | 0.30 | 0.10 |

A box contains parts that all come from **one** machine, but the label is lost.

(a) You test one part from the box and it is defective. What is the probability that the box came from machine B? Use the table method of the marble game: **prior × likelihood, then divide by the sum.**
(b) A second part from the same box is also defective. Update again, using your answer to (a) as the new prior (slide "The fourth draw is blue: today's posterior is the new prior"). Check that you get the same answer by updating once with both parts (likelihood 0.02² for A and 0.10² for B, because parts are independent given the machine).
(c) What is the probability that a **third** part from this box is defective? Average over the machines, as on the slide "After B W B, what is the chance the next draw is blue?".
(d) Check (a) by simulation: simulate 100,000 boxes, test one part from each, keep only the boxes whose tested part was defective, and find the share that came from machine B.

In [ ]:
prior <- c(A = 0.7, B = 0.3)      # share of parts from each machine
p_def <- c(A = 0.02, B = 0.10)    # P(defective | machine): the likelihood of "defective"

In [ ]:
# (a) post1 <- prior * p_def / sum(prior * p_def)
# (b) post2 <- post1 * p_def / sum(post1 * p_def); compare with prior * p_def^2 / sum(prior * p_def^2)
# (c) sum(post2 * p_def)
# (d) machine   <- sample(c("A", "B"), 100000, replace = TRUE, prob = prior)
#     defective <- runif(100000) < p_def[machine]    # p_def[machine] picks 0.02 or 0.10 for each box
#     mean(machine[defective] == "B")

<details><summary><b>Hint</b> (click to open)</summary>

In (d), `defective` is TRUE or FALSE for each box, so `machine[defective]` keeps only the machines of the boxes with a defective part.

</details>

**Your answer:** *(write here)*

### B2. Densities are not probabilities (25 min)

*Background: the slides "A PDF spreads probability out", "The CDF works for both kinds" and "A density is not a probability" (afternoon).*

The waiting time *T* (minutes) between two requests at a shared lab instrument is modelled as **exponential with mean 6 minutes**, i.e. rate 1/6 per minute. In R: `dexp(t, rate)` is the density, `pexp(t, rate)` = P(T ≤ t), `qexp(prob, rate)` the quantile.

(a) Evaluate the density at *t* = 0 and *t* = 3. What are its units?
(b) Compute P(*T* < 3), P(2 < *T* < 4) and P(*T* = 3).
(c) Find the median waiting time and compare it with the mean of 6 minutes.
(d) A different quantity *U* is uniform on [0, 0.25] hours. What is its density, and why is a density larger than 1 not a contradiction?

In [ ]:
rate <- 1/6

In [ ]:
# (a) dexp(c(0, 3), rate)
# (b) pexp(3, rate);  pexp(4, rate) - pexp(2, rate)
# (c) qexp(0.5, rate)
# (d) dunif(0.1, 0, 0.25)

<details><summary><b>Hint</b> (click to open)</summary>

Probabilities are **areas** under the density: P(a < T < b) = F(b) − F(a), where F is `pexp`. The probability of any single exact value is 0.

</details>

**Your answer:** *(write here)*

### B3. Expectation, variance and a first prediction (25 min)

*Background: the slides "A PMF puts probability on each value: two dice", "Notation conventions" (E[X] and Var(X)) and "Four letters in front of every distribution", afternoon.*

The **expected value** (long-run average) and the **variance** (expected squared distance from the expected value) of a discrete random variable X are computed from its PMF p(x) = P(X = x):

$$E[X] = \sum_x x\, p(x), \qquad \text{Var}(X) = \sum_x (x - E[X])^2\, p(x).$$

For a binomial count with n trials and success probability p they simplify to E[X] = np and Var(X) = np(1 − p).

(a) Let *X* be the number of defective parts in a box of 5 parts from machine B (each defective with probability 0.10, independently), so X ~ Binomial(5, 0.10). Tabulate the PMF with `dbinom()`, compute E[X] and Var(X) **from the table** with the sums above, and compare with np and np(1 − p). Check both by simulation with `rbinom()`.
(b) Back to the weighing data. Give an informal central prediction and a plausible range for (i) **the next weighing by O4** and (ii) **the first weighing by a new operator** who has not been observed. Explain why (ii) should be wider. This is informal reasoning: proper Bayesian predictions for such questions come on Days 2 and 4.

In [ ]:
x  <- 0:5
px <- dbinom(x, 5, 0.10)     # the PMF of Binomial(5, 0.10)
tibble(x = x, px = round(px, 4))

In [ ]:
# (a) E <- sum(x * px);  V <- sum((x - E)^2 * px);  compare with 5 * 0.1 and 5 * 0.1 * 0.9
#     sims <- rbinom(100000, 5, 0.1);  mean(sims);  var(sims)
# (b) use your per-operator table from A2

<details><summary><b>Hint</b> (click to open)</summary>

For (b)(ii), think about two sources of variation: where a new operator's own average might lie (look at how much the five operator means differ) *and* the scatter of single weighings around that average.

</details>

**Your answer:** *(write here)*

### Reflection B (optional)

In 80–120 words: explain the difference between P(B | defective) and P(defective | B) using the numbers in B1 (slide "Conditional probability: restrict, then re-scale"), and say in one sentence why the predictive probability in B1(c) lies between the two machines' defect rates.

**Your answer:** *(write here)*

## C. Optional challenges

**C1.** In B1, how many defective parts in a row are needed before P(B | data) exceeds 0.99? Find the answer with a short loop that repeats the update of B1(b). Why does the answer depend on the prior 0.3?

**C2.** For each operator, compute the mean and the rough **standard error** SD / √n, which measures how precisely the mean of n readings is known. Is O4's mean clearly different from 0 mg by this measure? Why does this say nothing about a *new* operator?

**C3.** Waiting times are recorded in minutes, but occasionally one value is recorded in **seconds** by mistake (60 times too large). Simulate 5,000 datasets of 9 exponential waiting times (mean 6 minutes); in 10% of the datasets multiply one randomly chosen value by 60. Compare how much the **mean** and the **median** of the 9 values vary across datasets. Which summary is more stable?

In [ ]:
# Optional challenge work

**Your answer:** *(write here)*

## Make-up checklist

- [ ] A1–A3 and B1–B3 · [ ] in-class exercises (`../day-1-exercises-morning.ipynb` and `../day-1-exercises-afternoon.ipynb`)
- [ ] The notebook runs from a fresh kernel; plots have labelled axes with units
- [ ] Every answer cell contains your own explanation, not only numbers
- [ ] I can explain every line of my code